# Sequence Length Hidden State Debug


In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

from pfns.experiments.model_benchmarks.model_registry import get_models_from_names
from pfns.experiments.model_benchmarks.seq_len_debug_utils import (
    collect_state_renorm_scales,
    plot_avg_metric_per_layer_per_head,
    plot_metric_layer_seqlen_heatmap,
    plot_recurrent_metric_per_layer,
    plot_recurrent_metric_per_head,
    run_hidden_state_tracking,
    summarize_hidden_state_by_seqlen
)
from pfns.utils import get_default_device

plt.rcParams['figure.dpi'] = 300

## Config


In [ ]:
EXPERIMENT = {
    'name': 'seq_len_hidden_state_debug',
    'num_classes': 5,
    'num_features': 10,
    'num_test_samples': 100,
    'num_repetitions': 50,
    'data_generation_seed': 42,
    'seqlen_list': [250, 500, 1_000, 2_000, 4_000, 8_000, 16_000, 32_000, 64_000, 128_000],
}

MODEL_NAMES = [
    'Linear_Attention_Non_Causal',
    'Linear_Attention_Comb_ST',
    'Linear_Attention_Non_Causal_fro_norm',
    'Linear_Attention_Comb_ST_fro_norm',
]

TRAINING_CONTEXT_LENGTH = 1_000
REFERENCE_CONTEXT_LENGTH = max(EXPERIMENT['seqlen_list'])
REFERENCE_SUBSPACE_RANK = 8

# Toggle between the original per-run traces and seq-len violin distributions.
HIDDEN_STATE_PLOT_MODE = 'violin'  # 'violin' or 'individual_runs'
HIDDEN_STATE_DISTRIBUTION_ALPHA = 0.3
HIDDEN_STATE_DISTRIBUTION_WIDTH_FRAC = 0.4

def resolve_graphics_dir():
    for base in (Path.cwd(), *Path.cwd().parents):
        candidate = base / 'PFNs' / 'graphics'
        if candidate.exists():
            return candidate
    return Path.cwd() / 'PFNs' / 'graphics'

HIDDEN_STATE_GRAPHICS_DIR = resolve_graphics_dir() / 'hidden_state_debug'
HIDDEN_STATE_GRAPHICS_DIR.mkdir(parents=True, exist_ok=True)
HIDDEN_STATE_FIGURE_FORMATS = ('pdf',)

device = str(get_default_device())
models_to_compare = get_models_from_names(MODEL_NAMES)

print(f'Device: {device}')
print(f'Models: {list(models_to_compare)}')
print(f'Repetitions: {EXPERIMENT["num_repetitions"]}')
print(f'Seq lens: {EXPERIMENT["seqlen_list"]}')
print(f'Hidden-state plot mode: {HIDDEN_STATE_PLOT_MODE}')
print(f'Hidden-state plots: {HIDDEN_STATE_GRAPHICS_DIR}')


## Run Hidden-State Tracking


In [ ]:
from pathlib import Path

import pandas as pd

from pfns.experiments.model_benchmarks.hashing import experiment_payload_hash
from pfns.experiments.model_benchmarks.model_registry import functional_model_config

HIDDEN_STATE_CACHE_DIR = Path("hidden_state_cache")
HIDDEN_STATE_CACHE_DIR.mkdir(parents=True, exist_ok=True)

HIDDEN_STATE_CACHE_OVERWRITE = False
HIDDEN_STATE_TRACKING_METRICS_VERSION = 'adjacent_readout_metrics_v1'
REQUIRED_HIDDEN_STATE_COLUMNS = {
    'output_norm',
    'output_cosine_to_reference',
    'state_cosine_to_previous',
    'output_cosine_to_previous',
}

hidden_state_cache_key = experiment_payload_hash(
    experiment_payload={
        "experiment": EXPERIMENT,
        "reference_seqlen": REFERENCE_CONTEXT_LENGTH,
        "reference_subspace_rank": REFERENCE_SUBSPACE_RANK,
        "tracking_metrics_version": HIDDEN_STATE_TRACKING_METRICS_VERSION,
        "models_to_compare": {
            model_name: functional_model_config(model_config)
            for model_name, model_config in models_to_compare.items()
        },
    }
)
hidden_state_cache_path = (
    HIDDEN_STATE_CACHE_DIR / f"{EXPERIMENT['name']}_{hidden_state_cache_key}.pkl"
)

if hidden_state_cache_path.exists() and not HIDDEN_STATE_CACHE_OVERWRITE:
    hidden_state_df = pd.read_pickle(hidden_state_cache_path)
    missing_cached_columns = REQUIRED_HIDDEN_STATE_COLUMNS - set(hidden_state_df.columns)
    if missing_cached_columns:
        print(f"Ignoring cached hidden_state_df with missing columns: {sorted(missing_cached_columns)}")
        hidden_state_df = None
    else:
        print(f"Loaded hidden_state_df from cache: {hidden_state_cache_path}")
else:
    hidden_state_df = None

if hidden_state_df is None:
    hidden_state_df = run_hidden_state_tracking(
        experiment=EXPERIMENT,
        models_to_compare=models_to_compare,
        device=device,
        reference_seqlen=REFERENCE_CONTEXT_LENGTH,
        reference_subspace_rank=REFERENCE_SUBSPACE_RANK,
    )
    hidden_state_df.to_pickle(hidden_state_cache_path)
    print(f"Saved hidden_state_df to cache: {hidden_state_cache_path}")

print("hidden_state_df rows:", len(hidden_state_df))
hidden_state_df.head()


## Summaries and Plots


In [ ]:
summary_df = summarize_hidden_state_by_seqlen(hidden_state_df)

print('summary_df rows:', len(summary_df))
summary_df.head()

### Metric Guide
- `abs_max_mean`: average largest absolute value in each recurrent-state head matrix.
- `frobenius_norm_mean`: per-head Frobenius norm of `recurrent_state` matrices.
- `spectral_norm_mean`: per-head spectral norm (largest singular value) of `recurrent_state` matrices.
- `effective_rank_mean`: DeltaProduct-style effective rank `exp(H(sigma / ||sigma||_1))` of each recurrent-state head matrix.
- `stable_rank_mean`: stable rank `||A||_F^2 / ||A||_2^2` of each recurrent-state head matrix.
- `state_cosine_to_reference`: cosine similarity between `vec(KV_state_L)` and `vec(KV_state_reference)` for the same generated dataset prefix.
- `state_top_subspace_to_reference`: average overlap of the top left/right singular-vector subspaces of `KV_state_L` and `KV_state_reference`; 1 means the dominant matrix subspaces match, even if individual singular-vector signs differ.
- `output_norm`: Frobenius norm of the per-layer per-head readout tensor `q @ M` before output projection.
- `output_cosine_to_reference`: cosine similarity between `q @ M_L` and `q @ M_reference` on the shared sequence prefix.

In [ ]:
all_tensors = sorted(summary_df['tensor_name'].astype(str).unique().tolist())
print(f'Recurrent-state tensors available: {len(all_tensors)}')
for name in all_tensors:
    print(' ', name)

In [ ]:
shared_plot_kwargs = {
    'plot_mode': HIDDEN_STATE_PLOT_MODE,
    'distribution_alpha': HIDDEN_STATE_DISTRIBUTION_ALPHA,
    'distribution_width_frac': HIDDEN_STATE_DISTRIBUTION_WIDTH_FRAC,
    'show_title': True,
    'show_suptitle': False,
    'include_group_in_title': False,
    'log_y': True
}
shared_save_kwargs = {
    'save_dir': HIDDEN_STATE_GRAPHICS_DIR,
    'save_formats': HIDDEN_STATE_FIGURE_FORMATS,
}

hidden_state_df_plot = hidden_state_df.copy()
HIDDEN_STATE_DISPLAY_NAME_OVERRIDES = {
    'Linear_Attention_Comb_ST': 'Causal Linear Attention (unnormalised)',
    'Linear_Attention_Non_Causal': 'Non-Causal Linear Attention (unnormalised)',
    'Linear_Attention_Comb_ST_fro_norm': 'Causal Linear Attention with Frobenius Norm',
    'Linear_Attention_Non_Causal_fro_norm': 'Non-Causal Linear Attention with Frobenius Norm',
}
if 'display_name' not in hidden_state_df_plot.columns:
    hidden_state_df_plot['display_name'] = hidden_state_df_plot['model'].astype(str)
hidden_state_df_plot['display_name'] = (
    hidden_state_df_plot['model'].astype(str).map(HIDDEN_STATE_DISPLAY_NAME_OVERRIDES)
    .fillna(hidden_state_df_plot['display_name'])
)

plot_specs = [
    # (
    #     hidden_state_df_plot,
    #     'abs_max',
    #     plot_recurrent_metric_per_head,
    #     {
    #         'title_prefix': 'Recurrent-state abs max vs sequence length',
    #         'training_context_length': TRAINING_CONTEXT_LENGTH,
    #         **shared_plot_kwargs,
    #     },
    # ),
    # (
    #     hidden_state_df_plot,
    #     'state_cosine_to_reference',
    #     plot_recurrent_metric_per_layer,
    #     {
    #         'title_prefix': f'Recurrent-state cosine similarity to {TRAINING_CONTEXT_LENGTH} in-context tokens',
    #         'training_context_length': TRAINING_CONTEXT_LENGTH,
    #         'plot_mode': HIDDEN_STATE_PLOT_MODE,
    #         'distribution_alpha': HIDDEN_STATE_DISTRIBUTION_ALPHA,
    #         'distribution_width_frac': HIDDEN_STATE_DISTRIBUTION_WIDTH_FRAC,
    #         'show_title': True,
    #         'show_suptitle': False,
    #         'legend_title': 'State Cosine Similarity to Reference',
    #         'log_y': False,
    #     },
    # ),
    (
        hidden_state_df_plot,
        'frobenius_norm',
        plot_recurrent_metric_per_head,
        {
            'title_prefix': 'Recurrent-state Frobenius norm drift',
            'training_context_length': TRAINING_CONTEXT_LENGTH,
            **shared_plot_kwargs,
        },
    ),
    (
        hidden_state_df_plot,
        'spectral_norm',
        plot_recurrent_metric_per_head,
        {
            'title_prefix': 'Recurrent-state spectral norm vs sequence length',
            'training_context_length': TRAINING_CONTEXT_LENGTH,
            **shared_plot_kwargs,
        },
    ),
    (
        hidden_state_df_plot,
        'effective_rank',
        plot_recurrent_metric_per_layer,
        {
            'title_prefix': 'Recurrent-state effective rank vs sequence length',
            'training_context_length': TRAINING_CONTEXT_LENGTH,
            **shared_plot_kwargs,
        },
    ),
    (
        hidden_state_df_plot,
        'stable_rank',
        plot_recurrent_metric_per_layer,
        {
            'title_prefix': 'Recurrent-state stable rank vs sequence length',
            'training_context_length': TRAINING_CONTEXT_LENGTH,
            **shared_plot_kwargs,
        },
    ),
    # (
    #     hidden_state_df_plot,
    #     'kv_state_norm',
    #     plot_recurrent_metric_per_layer,
    #     {
    #         'title_prefix': 'KV-state norm vs sequence length',
    #         'training_context_length': TRAINING_CONTEXT_LENGTH,
    #         **shared_plot_kwargs,
    #     },
    # ),
    # (
    #     hidden_state_df_plot,
    #     'k_sum_norm',
    #     plot_recurrent_metric_per_layer,
    #     {
    #         'title_prefix': 'K-sum norm vs sequence length',
    #         'training_context_length': TRAINING_CONTEXT_LENGTH,
    #         **shared_plot_kwargs,
    #     },
    # ),
    # (
    #     hidden_state_df_plot,
    #     'joint_hidden_state_norm',
    #     plot_recurrent_metric_per_layer,
    #     {
    #         'title_prefix': 'Joint hidden-state norm vs sequence length',
    #         'training_context_length': TRAINING_CONTEXT_LENGTH,
    #         **shared_plot_kwargs,
    #     },
    # ),
    # (
    #     hidden_state_df_plot,
    #     'kv_over_ksum_ratio',
    #     plot_recurrent_metric_per_layer,
    #     {
    #         'title_prefix': 'KV-over-K-sum ratio vs sequence length',
    #         'training_context_length': TRAINING_CONTEXT_LENGTH,
    #         **shared_plot_kwargs,
    #     },
    # ),
]

for plot_df, metric, plot_fn, plot_kwargs in plot_specs:
    if metric not in plot_df.columns:
        continue
    metric_values = pd.to_numeric(plot_df[metric], errors='coerce')
    if not metric_values.notna().any():
        continue
    plot_fn(
        plot_df,
        metric=metric,
        filename_prefix=metric,
        **shared_save_kwargs,
        **plot_kwargs,
    )

HEATMAP_SCALE_REFERENCE_GAP = 'linear' # 'log_reference_gap' or 'linear'

plot_metric_layer_seqlen_heatmap(
    hidden_state_df_plot,
    metric='state_cosine_to_reference',
    title_prefix=f'Average recurrent-state cosine similarity to {TRAINING_CONTEXT_LENGTH} in-context tokens',
    training_context_length=TRAINING_CONTEXT_LENGTH,
    cmap='magma',
    color_scale=HEATMAP_SCALE_REFERENCE_GAP,
)

plot_metric_layer_seqlen_heatmap(
    hidden_state_df_plot,
    metric='state_top_subspace_to_reference',
    title_prefix=f'Average recurrent-state top-subspace similarity to {TRAINING_CONTEXT_LENGTH} in-context tokens',
    training_context_length=TRAINING_CONTEXT_LENGTH,
    cmap='magma',
    color_scale=HEATMAP_SCALE_REFERENCE_GAP,
)


print('Per-head metric summaries across sequence lengths:')
for metric, title_prefix in [
    ('effective_rank', 'Effective rank per layer'),
    ('stable_rank', 'Stable rank per layer'),
]:
    if metric not in hidden_state_df_plot.columns:
        continue
    metric_values = pd.to_numeric(hidden_state_df_plot[metric], errors='coerce')
    if not metric_values.notna().any():
        continue
    plot_avg_metric_per_layer_per_head(
        hidden_state_df_plot,
        metric=metric,
        title_prefix=title_prefix,
        show_title=True,
        show_suptitle=False,
        filename_prefix=f'{metric}_avg_per_layer_per_head',
        **shared_save_kwargs,
    )


### Split Frobenius Norm Summary


In [ ]:
import matplotlib.lines as mlines
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pfns.experiments.model_benchmarks.plotting import apply_pretraining_split_background

DEBUG_NORM_MODEL_GROUPS = {
    'Causal variants': [
        'Linear_Attention_Comb_ST',
        'Linear_Attention_Comb_ST_fro_norm',
    ],
    'Non-causal variants': [
        'Linear_Attention_Non_Causal',
        'Linear_Attention_Non_Causal_fro_norm',
    ],
}
DEBUG_NORM_MODEL_ORDER = [
    'Linear_Attention_Comb_ST',
    'Linear_Attention_Comb_ST_fro_norm',
    'Linear_Attention_Non_Causal',
    'Linear_Attention_Non_Causal_fro_norm',
]
DEBUG_NORM_MODEL_STYLES = {
    'Linear_Attention_Comb_ST': {'color': '#00796B', 'marker': 'o'},
    'Linear_Attention_Comb_ST_fro_norm': {'color': '#D55E00', 'marker': 's'},
    'Linear_Attention_Non_Causal': {'color': '#00796B', 'marker': 'o'},
    'Linear_Attention_Non_Causal_fro_norm': {'color': '#D55E00', 'marker': 's'},
}
DEBUG_NORM_DISPLAY_NAMES = {
    'Linear_Attention_Comb_ST': 'Causal',
    'Linear_Attention_Comb_ST_fro_norm': 'Causal + Frobenius norm',
    'Linear_Attention_Non_Causal': 'Non-causal',
    'Linear_Attention_Non_Causal_fro_norm': 'Non-causal + Frobenius norm',
}
DEBUG_NORM_FIGSIZE = (12.8, 4.8)

if 'hidden_state_df_plot' not in globals() or hidden_state_df_plot.empty:
    raise RuntimeError('No hidden_state_df_plot found. Run the hidden-state tracking and plotting setup cells first.')
if 'frobenius_norm' not in hidden_state_df_plot.columns:
    raise RuntimeError('hidden_state_df_plot has no frobenius_norm column.')

required_norm_models = [
    model
    for group_models in DEBUG_NORM_MODEL_GROUPS.values()
    for model in group_models
]
norm_plot_df = hidden_state_df_plot[
    hidden_state_df_plot['model'].astype(str).isin(required_norm_models)
].copy()
norm_plot_df['frobenius_norm'] = pd.to_numeric(norm_plot_df['frobenius_norm'], errors='coerce')
norm_plot_df = norm_plot_df[np.isfinite(norm_plot_df['frobenius_norm'])].copy()

missing_norm_models = sorted(set(required_norm_models) - set(norm_plot_df['model'].astype(str).unique()))
if missing_norm_models:
    print(f'Missing requested models in hidden_state_df_plot: {missing_norm_models}')
if norm_plot_df.empty:
    raise RuntimeError('No finite Frobenius-norm rows found for the requested models.')

norm_summary = (
    norm_plot_df.groupby(['model', 'seqlen'], observed=True)['frobenius_norm']
    .agg(median='median', q25=lambda values: values.quantile(0.25), q75=lambda values: values.quantile(0.75))
    .reset_index()
    .sort_values(['model', 'seqlen'])
)

finite_norm_values = norm_summary[['q25', 'q75']].to_numpy(dtype=float).ravel()
finite_norm_values = finite_norm_values[np.isfinite(finite_norm_values) & (finite_norm_values > 0.0)]
ylim = None
if finite_norm_values.size:
    ylim = (max(float(finite_norm_values.min()) / 1.2, 1e-12), float(finite_norm_values.max()) * 1.2)
positive_x_values = norm_summary['seqlen'].to_numpy(dtype=float, copy=False)
positive_x_values = positive_x_values[np.isfinite(positive_x_values) & (positive_x_values > 0.0)]

fig, axes = plt.subplots(1, 2, figsize=DEBUG_NORM_FIGSIZE, dpi=300, sharex=True, sharey=True)
fig.patch.set_facecolor('white')

for ax, (panel_title, panel_models) in zip(axes, DEBUG_NORM_MODEL_GROUPS.items()):
    ax.set_facecolor('#fbfbfb')
    for model in panel_models:
        sub = norm_summary[norm_summary['model'].astype(str) == model]
        if sub.empty:
            continue
        style = DEBUG_NORM_MODEL_STYLES[model]
        ax.plot(
            sub['seqlen'],
            sub['median'],
            color=style['color'],
            marker=style['marker'],
            linewidth=2.0,
            markersize=5.2,
            label=DEBUG_NORM_DISPLAY_NAMES[model],
        )
        ax.fill_between(
            sub['seqlen'],
            sub['q25'],
            sub['q75'],
            color=style['color'],
            alpha=0.12,
            linewidth=0,
        )
    ax.set_xscale('log')
    ax.set_yscale('log')
    if positive_x_values.size:
        ax.set_xlim(float(positive_x_values.min()), float(positive_x_values.max()))
    if ylim is not None:
        ax.set_ylim(*ylim)
    apply_pretraining_split_background(ax, boundary=float(TRAINING_CONTEXT_LENGTH), boundary_label='Train context')
    ax.set_title(panel_title)
    ax.set_xlabel('Sequence length')
    ax.grid(True, which='major', ls='-', alpha=0.16)
    ax.grid(True, which='minor', ls='-', alpha=0.06)
axes[0].set_ylabel('Recurrent-state Frobenius norm')

legend_handles = [
    mlines.Line2D(
        [], [],
        color=DEBUG_NORM_MODEL_STYLES[model]['color'],
        marker=DEBUG_NORM_MODEL_STYLES[model]['marker'],
        linestyle='-',
        linewidth=2.0,
        markersize=5.2,
        label=DEBUG_NORM_DISPLAY_NAMES[model],
    )
    for model in DEBUG_NORM_MODEL_ORDER
]
fig.legend(
    handles=legend_handles,
    loc='lower center',
    bbox_to_anchor=(0.5, -0.02),
    ncol=2,
    fontsize=8.4,
    frameon=True,
    facecolor='white',
    edgecolor='#d4d4d4',
    framealpha=1.0,
    handlelength=2.1,
    columnspacing=1.2,
)
fig.tight_layout(rect=(0, 0.14, 1, 1))

split_norm_paths = []
for fmt in HIDDEN_STATE_FIGURE_FORMATS:
    path = HIDDEN_STATE_GRAPHICS_DIR / f'frobenius_norm_causal_noncausal_split.{fmt.lstrip(".")}'
    fig.savefig(path, bbox_inches='tight')
    split_norm_paths.append(path)
display(fig)
print('Saved split Frobenius-norm summary figures:')
for path in split_norm_paths:
    print(f' - {path}')


In [ ]:
prefix_to_reference_summary = (
    hidden_state_df_plot[
        hidden_state_df_plot['seqlen'].le(REFERENCE_CONTEXT_LENGTH)
        & hidden_state_df_plot['state_cosine_to_reference'].notna()
    ]
    .groupby(['model', 'layer_idx', 'head_idx'], observed=True)['state_cosine_to_reference']
    .mean()
    .reset_index(name='sampled_to_reference_cosine_mean')
    .sort_values(['model', 'layer_idx', 'head_idx'])
)

prefix_to_reference_summary.head()


In [ ]:
state_renorm_scale_df = collect_state_renorm_scales(
    models_to_compare=models_to_compare,
    device=device,
)

plot_avg_metric_per_layer_per_head(
    state_renorm_scale_df,
    metric='state_renorm_scale',
    title_prefix='Learnt Frobenius renormalisation scale per layer',
    show_title=True,
    show_suptitle=False,
    filename_prefix='state_renorm_scale_avg_per_layer_per_head',
    save_dir=HIDDEN_STATE_GRAPHICS_DIR,
    save_formats=HIDDEN_STATE_FIGURE_FORMATS,
)

state_renorm_scale_df.head()
